# Chapter 5 &mdash; Cross-Checking a Specification: $L_{eqc}$

**Concept 2 of the Chapter 5 decomposition:** *Cross-Checking a Specification: the Equal-Changes Language $L_{eqc}$*

Four specifications of one language, disagreeing &mdash; and numeric order as the systematic tie-breaker.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5-DFADsg/Concept-Cross-Checking-Leqc/Concept-Cross-Checking-Leqc.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


$L_{eqc}$: strings over $\{0,1\}$ with an **equal number of changes** from 0 to 1 and
from 1 to 0.

Four people write four specifications and they **disagree** &mdash; usually about
$\varepsilon$, single symbols, and whether "change" means adjacent-pair.

The systematic cure is **enumeration in numeric order**: `nthnumeric(i)` lists
$\varepsilon, 0, 1, 00, 01, 10, 11, 000, \dots$ so the short strings, where the
disagreements live, come **first**.

## 2. Definitions

### The language, and a change-counter

In [ ]:
def changes(s):
    """pairs (0->1) and (1->0) in adjacent positions"""
    up   = sum(1 for a, b in zip(s, s[1:]) if a == '0' and b == '1')
    down = sum(1 for a, b in zip(s, s[1:]) if a == '1' and b == '0')
    return up, down

def in_Leqc(s):
    u, d = changes(s)
    return u == d

### A rival specification someone might write

In [ ]:
def in_Leqc_rival(s):
    """'starts and ends with the same symbol' -- is that the same language?"""
    return s != '' and s[0] == s[-1]

### Numeric order puts the short strings first

In [ ]:
def numeric(n, sigma=['0', '1']):
    return [nthnumeric(i, sigma) for i in range(n)]

<!-- nav-strip -->

---

&larr;&nbsp;[Ch5&nbsp;1.&nbsp;Four Ways to Specify a Language, and Why You Need More Than One](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5-DFADsg/Concept-Four-Ways-To-Specify/Concept-Four-Ways-To-Specify.ipynb) &nbsp;&middot;&nbsp; [**Chapter 5** index](https://github.com/ganeshutah/Jove/blob/master/Chapter5-DFADsg/README.md) &nbsp;&middot;&nbsp; [Ch5&nbsp;3.&nbsp;Best Practices: Mnemonic State Names and Documented Transitions](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5-DFADsg/Concept-Mnemonic-State-Names/Concept-Mnemonic-State-Names.ipynb)&nbsp;&rarr;

---

## 3. Tests

Numeric order: $\varepsilon$ first, then length 1, then length 2 &mdash; exactly where bugs hide.

In [ ]:
print(numeric(16))
assert nthnumeric(0, ['0','1']) == ''
print("\nNote nthnumeric needs a LIST, not a set -- a set has no order.")

The two specifications **disagree**, and numeric order finds the shortest witness fast.

In [ ]:
diff = [s for s in numeric(64) if in_Leqc(s) != in_Leqc_rival(s)]
print("shortest disagreements :", diff[:6])
assert '' in diff
print("\nepsilon: 0 changes each way -> in L_eqc; but '' has no first symbol -> not in the rival.")

Repairing the rival makes the two views agree &mdash; the cross-check is now informative.

In [ ]:
def in_Leqc_fixed(s):
    return s == '' or s[0] == s[-1]

diff = [s for s in numeric(512) if in_Leqc(s) != in_Leqc_fixed(s)]
print("disagreements after repair :", diff)
assert not diff
print("\nEqual up-changes and down-changes <=> first symbol equals last. Now build the DFA.")

The DFA, tested against the numeric-order enumeration.

In [ ]:
Leqc = md2mc('''DFA
IF  : 0 -> F0
IF  : 1 -> F1
F0  : 0 -> F0
F0  : 1 -> S1
F1  : 1 -> F1
F1  : 0 -> S0
S0  : 0 -> S0
S0  : 1 -> F1
S1  : 1 -> S1
S1  : 0 -> F0
''')
assert all(accepts_dfa(Leqc, s) == in_Leqc(s) for s in numeric(512))
print("DFA agrees with the specification on the first 512 strings in numeric order")

## 4. Exercises


1. Write a fifth specification of $L_{eqc}$ and cross-check it.
2. Why does numeric order beat random testing for finding specification bugs?
3. What does `nthnumeric(i, ['a','b','c'])` enumerate? Try it.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter5-DFADsg/Concept-Cross-Checking-Leqc')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')